# 02 — Train and Evaluate

End-to-end walkthrough: load data → train a model → generate probabilistic PV profiles → visualise uncertainty → compute metrics.

Three models are available via the `model` key in the config:

| Key | Description |
|---|---|
| `cddpm` | Conditional diffusion model — **paper main model** |
| `cvae`  | Conditional VAE — probabilistic benchmark |
| `qr`    | Quantile regression CNN — probabilistic benchmark |

> ⚡ **Quick demo mode:** `N_EPOCHS` below is set to a small value so the notebook runs in a few minutes. For full paper results, use `n_epochs=2000` (cDDPM/cVAE) or `n_epochs=1000` (QR) on a GPU.


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, "../src")

from main import run, BEST_CONFIGS, DROP_CUSTOMERS, load_data
from utils import set_seed, split_train_test_tensors, refine_samples

DATA_DIR = "../data"

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 13,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.dpi": 110,
})

# ── Demo flag ──────────────────────────────────────────────
# Set N_EPOCHS small for a quick run. Use None to keep the paper values.
N_EPOCHS = 100   # ← change to None for full training
# ──────────────────────────────────────────────────────────


## 1. Choose and configure a model

In [ ]:
MODEL = "cddpm"   # ← change to "cvae" or "qr" to run a benchmark

config = {
    **BEST_CONFIGS[MODEL],
    "seed": 0,
}

if N_EPOCHS is not None:
    config["n_epochs"] = N_EPOCHS

print(f"Model  : {config['model'].upper()}")
print(f"Epochs : {config['n_epochs']}")
print(f"Device : {'CUDA' if torch.cuda.is_available() else 'CPU (training may be slow)'}")


## 2. Load data

In [ ]:
exp_data, capacity_data = load_data(config)

pv_train  = exp_data["pv_train"]
pv_test   = exp_data["pv_test"]

print(f"Training set : {pv_train.shape[0]:,} daily profiles")
print(f"Test set     : {pv_test.shape[0]:,} daily profiles")
print(f"Profile length: {pv_train.shape[1]} time steps")


## 3. Train

In [ ]:
from main import _train
import time

set_seed(config["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
config["device"] = device

t0 = time.time()
model = _train(config, exp_data)
print(f"\nDone in {time.time() - t0:.1f}s")


## 4. Generate probabilistic samples for a single day

Let's pick one test customer and one day, then generate multiple plausible PV profiles.


In [ ]:
from main import _generate

# Pick a specific test sample
SAMPLE_IDX = 50   # ← change to explore different customers / days

real_pv = exp_data["pv_test"][SAMPLE_IDX]
conditions_test = {k: v[SAMPLE_IDX] for k, v in exp_data["conditions_test"].items()}
customer_id = exp_data["test_customers"][SAMPLE_IDX]
date        = exp_data["test_dates"][SAMPLE_IDX]

print(f"Customer: {customer_id}  |  Date: {date}")

# Generate samples  →  [S, 48] or [Q, 48] for QR
generated = _generate(model, conditions_test, config)

# Denormalise using capacity
cap_pred = capacity_data.loc[customer_id, "cap_pred"]
cap_real = capacity_data.loc[customer_id, "cap_real"]
max_exp  = capacity_data.loc[customer_id, "max_export"]
gi_cond  = conditions_test.get("gi") if "gi" in config["conditions"] else None

refined = refine_samples(generated, gi_cond, None, cap_pred, max_exp)  # kW
real_kw = real_pv * cap_real

print(f"Generated shape : {generated.shape}  (S samples × T time steps)")
print(f"Real PV peak    : {real_kw.max():.2f} kW")
print(f"Median pred peak: {refined.median(dim=0).values.max():.2f} kW")


## 5. Visualise uncertainty

In [ ]:
t = np.arange(48) * 0.5   # hours

refined_np = refined.numpy()   # [S, 48]
real_np    = real_kw.numpy()

p5,  p25, p50, p75, p95 = np.percentile(refined_np, [5, 25, 50, 75, 95], axis=0)

fig, ax = plt.subplots(figsize=(10, 4))

ax.fill_between(t, p5, p95, alpha=0.2, color="#3498DB", label="90% PI")
ax.fill_between(t, p25, p75, alpha=0.35, color="#3498DB", label="50% PI")
ax.plot(t, p50, color="#2980B9", linewidth=2, label="Median prediction")
ax.plot(t, real_np, color="#E74C3C", linewidth=1.8, linestyle="--", label="Ground truth")

ax.set_xlabel("Hour of day")
ax.set_ylabel("PV generation (kW)")
ax.set_title(f"Probabilistic disaggregation — Customer {customer_id}, {date} [{config['model'].upper()}]")
ax.set_xlim(0, 23.5)
ax.set_ylim(bottom=0)
ax.legend(fontsize=10)
fig.tight_layout()
plt.show()


In [ ]:
# Plot individual samples (to show diversity)
fig, ax = plt.subplots(figsize=(10, 4))

n_show = min(30, refined_np.shape[0])
for i in range(n_show):
    ax.plot(t, refined_np[i], color="#3498DB", alpha=0.15, linewidth=0.8)

ax.plot(t, p50,     color="#2980B9", linewidth=2,   label="Median")
ax.plot(t, real_np, color="#E74C3C", linewidth=1.8, linestyle="--", label="Ground truth")

ax.set_xlabel("Hour of day")
ax.set_ylabel("PV generation (kW)")
ax.set_title(f"Individual generated samples (n={n_show}) — [{config['model'].upper()}]")
ax.set_xlim(0, 23.5)
ax.set_ylim(bottom=0)
ax.legend(fontsize=10)
fig.tight_layout()
plt.show()


## 6. Run full evaluation

The `run()` function handles the full train-evaluate loop across all test customers (using seasonal stratification with `ratio_test=0.2` as in the paper).

> ⚠️ With `N_EPOCHS=100` the metrics will be worse than paper results. Set `N_EPOCHS=None` and use a GPU for reproduction.


In [ ]:
df_metrics = run(config)


In [ ]:
# Summary table
summary_cols = ["mase", "smape", "picp_50", "picp_90", "pinaw_50", "crps"]
available = [c for c in summary_cols if c in df_metrics.columns]

print(f"\n{'Metric':<15} {'Mean':>10} {'Std':>10}")
print("-" * 37)
for col in available:
    print(f"{col:<15} {df_metrics[col].mean():>10.4f} {df_metrics[col].std():>10.4f}")


## 7. Compare all three models (optional)

Run this cell to evaluate all three models sequentially and compare their average metrics. This will take significantly longer — consider running on a GPU.


In [ ]:
# ── Uncomment to run all three models ──────────────────────
# results = {}
# for model_name in ["cddpm", "cvae", "qr"]:
#     cfg = {**BEST_CONFIGS[model_name], "seed": 0}
#     if N_EPOCHS is not None:
#         cfg["n_epochs"] = N_EPOCHS
#     results[model_name] = run(cfg)
#
# # Compare
# comparison = pd.DataFrame({
#     name: df[available].mean()
#     for name, df in results.items()
# }).T
# print("\nModel comparison (average across test customers):")
# print(comparison.to_string(float_format="{:.4f}".format))


---

## Summary

| Step | What happened |
|---|---|
| Data loaded | Normalised Ausgrid data from `data/`, split into 107 train / 150 test customers |
| Model trained | Selected model learned to predict noise in the diffusion chain (cDDPM) or PV distribution directly (cVAE, QR) |
| Samples generated | Multiple plausible daily PV profiles produced per test day |
| Uncertainty quantified | Prediction intervals and probabilistic metrics computed |

For full paper results, retrain with `n_epochs=2000`, run 10 seeds, and average. See `src/main.py` for the exact configuration used.
